# DSA 2050 — Week 2 SQL Practical Lab

**SQL, Data Acquisition and Join Validation**  
Currency: Kenyan shillings (KSh). This notebook recreates the supplied lab data, saves the three source files beside the notebook, executes the SQL tasks in SQLite, and records the evidence-based answers. The duplicate customer row and unmatched order are intentionally retained until investigated.

## Part A — Create and save source data

In [1]:
import pandas as pd
import sqlite3
import json
from pathlib import Path

ROOT = Path.cwd()
customers = pd.DataFrame({
    "CustomerID": ["C001", "C002", "C003", "C004", "C004", "C005", "C006", "C007", "C008", "C009", "C010"],
    "CustomerName": ["Amina", "Brian", "Carol", "David", "David", "Esther", "Faith", "George", "Halima", "Ian", "Jane"],
    "Segment": ["Retail", "Corporate", "Retail", "SME", "SME", "Corporate", "SME", "Retail", "Corporate", "Retail", "SME"],
    "RegionCode": ["NRB", "MSA", "NRB", "KSM", "KSM", "NKR", "NRB", "MSA", "KSM", "NKR", "NRB"]
})
orders = pd.DataFrame({
    "OrderID": [f"O{i:03d}" for i in range(1, 16)],
    "CustomerID": ["C001", "C002", "C001", "C003", "C004", "C005", "C006", "C002", "C007", "C008", "C009", "C010", "C006", "C003", "C999"],
    "OrderAmount": [4500, 12500, 3200, 6800, 9100, 15000, 7600, 8300, 2500, 11000, 5400, 6200, 4300, 7200, 9900],
    "ProductCategory": ["Electronics", "Office", "Home", "Electronics", "Office", "Electronics", "Home", "Office", "Home", "Electronics", "Office", "Home", "Electronics", "Office", "Electronics"]
})
regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]
customers.to_csv(ROOT / "customers.csv", index=False)
orders.to_csv(ROOT / "orders.csv", index=False)
with open(ROOT / "regions.json", "w") as file:
    json.dump(regions, file, indent=4)
print("Saved customers.csv, orders.csv and regions.json to", ROOT)

Saved customers.csv, orders.csv and regions.json to /home/ubuntu/DSA2050-Week2-StudentID


## Part B — Inspect the sources
Reload from disk so the analysis uses the saved source files.

In [2]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
print("Customers shape:", customers.shape)
print("Orders shape:", orders.shape)
display(customers.head())
display(orders.head())

Customers shape: (11, 4)
Orders shape: (15, 4)


,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office


**Q1.** One row in `customers.csv` represents one customer record (intended grain: one row per unique CustomerID). The source violates that intended grain because C004 appears twice.

**Q2.** One row in `orders.csv` represents one individual order/transaction (one row per OrderID).

**Q3.** No. Customers are at customer-record grain; orders are at order/transaction grain. A customer may have multiple orders, so CustomerID is not unique in the order table.

## Part C — Detect key problems

In [3]:
duplicate_count = customers["CustomerID"].duplicated().sum()
duplicate_rows = customers[customers["CustomerID"].duplicated(keep=False)]
unmatched_ids = sorted(set(orders["CustomerID"]) - set(customers["CustomerID"]))
print("Duplicate CustomerID count:", duplicate_count)
display(duplicate_rows)
print("Order CustomerIDs missing from customers:", unmatched_ids)

Duplicate CustomerID count: 1


,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


Order CustomerIDs missing from customers: ['C999']


**Q4.** A duplicated CustomerID makes the supposed one-to-many relationship ambiguous. Each matching order can match multiple customer rows, multiplying output rows and repeating order amounts in aggregates.

**Q5.** `C999` appears in orders but not in customers.

**Q6.** It may represent a missing customer load, a keying error, or a legitimate unresolved transaction. Deleting it silently would lose an order and KSh 9,900 of sales; investigate and report it while preserving the source evidence.

## Part D — Create the SQLite database

In [4]:
conn = sqlite3.connect("retail_lab.db")
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
pd.read_sql_query("SELECT * FROM customers LIMIT 5;", conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


SQLite database `retail_lab.db` contains the original `customers` and `orders` tables.

## Part E — SELECT and WHERE

In [5]:
# E1: Select only requested order columns
query = "SELECT OrderID, CustomerID, OrderAmount FROM orders;"
display(pd.read_sql_query(query, conn))

# E2: Filter queries
retail_customers = pd.read_sql_query("SELECT * FROM customers WHERE Segment = 'Retail';", conn)
orders_over_8000 = pd.read_sql_query("SELECT * FROM orders WHERE OrderAmount > 8000;", conn)
electronics_orders = pd.read_sql_query("SELECT * FROM orders WHERE ProductCategory = 'Electronics';", conn)
electronics_over_8000 = pd.read_sql_query("SELECT * FROM orders WHERE ProductCategory = 'Electronics' AND OrderAmount > 8000;", conn)
print("Retail customer rows:", len(retail_customers))
print("Orders over KSh 8,000:", len(orders_over_8000))
print("Electronics orders:", len(electronics_orders))
print("Electronics orders over KSh 8,000:")
display(electronics_over_8000)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


Retail customer rows: 4
Orders over KSh 8,000: 6
Electronics orders: 6
Electronics orders over KSh 8,000:


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


## Part F — GROUP BY and category KPIs

In [6]:
category_kpis = pd.read_sql_query("""
SELECT ProductCategory,
       COUNT(*) AS NumberOfOrders,
       SUM(OrderAmount) AS TotalSales,
       AVG(OrderAmount) AS AverageOrderValue
FROM orders
GROUP BY ProductCategory
ORDER BY ProductCategory;
""", conn)
display(category_kpis)

,ProductCategory,NumberOfOrders,TotalSales,AverageOrderValue
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


**Q7.** Electronics generated the highest sales: KSh 51,500 across 6 orders, with an average order value of KSh 8,583.33. Office produced KSh 42,500 across 5 orders (average KSh 8,500), while Home was lowest at KSh 19,500 across 4 orders (average KSh 4,875).

## Part G — LEFT JOIN customers and orders

In [7]:
join_query = """
SELECT o.OrderID, o.CustomerID, c.CustomerName, c.Segment, o.OrderAmount
FROM orders o
LEFT JOIN customers c ON o.CustomerID = c.CustomerID;
"""
joined = pd.read_sql_query(join_query, conn)
print("Source order rows:", len(orders), "| Joined rows:", len(joined))
display(joined)

Source order rows: 15 | Joined rows: 16


,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


**Q8.** No. There are 15 order rows but 16 joined rows. O005 for C004 matches both duplicate C004 customer records, creating an extra joined row. The unmatched C999 remains as one row because this is a LEFT JOIN.

**Q9.** For order O015 / customer C999, `CustomerName` and `Segment` are NULL (shown as `None`/`NaN` in pandas).

## Part H — Reconcile totals

In [8]:
source_total = orders["OrderAmount"].sum()
joined_total = joined["OrderAmount"].sum()
print(f"Before join: KSh {source_total:,.0f}")
print(f"After join:  KSh {joined_total:,.0f}")
print(f"Difference:  KSh {joined_total-source_total:,.0f}")

Before join: KSh 113,500
After join:  KSh 122,600
Difference:  KSh 9,100


**Q10.** No. The source total is KSh 113,500 and the joined total is KSh 122,600, an overstatement of KSh 9,100.

**Q11.** C004 caused the duplication: its O005 order for KSh 9,100 was duplicated by the two right-side customer rows.

**Q12.** Reconciliation is a control that detects row multiplication, dropped records, or other integration errors before KPIs are trusted or reported.

## Part I — Correct the duplicate-key problem and rejoin

In [9]:
# Keep the first observed C004 record only after documenting the duplicate above.
customers_clean = customers.drop_duplicates(subset="CustomerID", keep="first").copy()
customers_clean.to_sql("customers_clean", conn, if_exists="replace", index=False)
clean_join_query = """
SELECT o.OrderID, o.CustomerID, c.CustomerName, c.Segment, c.RegionCode, o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c ON o.CustomerID = c.CustomerID;
"""
clean_joined = pd.read_sql_query(clean_join_query, conn)
print("Rows (source, clean join):", len(orders), len(clean_joined))
print("Sales (source, clean join):", orders.OrderAmount.sum(), clean_joined.OrderAmount.sum())
assert len(clean_joined) == len(orders)
assert clean_joined.OrderAmount.sum() == orders.OrderAmount.sum()

Rows (source, clean join): 15 15
Sales (source, clean join): 113500 113500


**Q13.** Yes. The corrected join has 15 rows, matching the 15 source orders, and KSh 113,500 total sales, matching the source. The left join still preserves the unmatched C999 order; removing the duplicate right-side key prevents row multiplication.

## Part J — Customer segment KPIs

In [10]:
segment_kpis = pd.read_sql_query("""
SELECT COALESCE(c.Segment, 'Unmatched / NULL') AS Segment,
       COUNT(*) AS Orders,
       SUM(o.OrderAmount) AS TotalSales,
       AVG(o.OrderAmount) AS AverageOrderValue
FROM orders o
LEFT JOIN customers_clean c ON o.CustomerID = c.CustomerID
GROUP BY c.Segment
ORDER BY c.Segment;
""", conn)
display(segment_kpis)

,Segment,Orders,TotalSales,AverageOrderValue
0,Unmatched / NULL,1,9900,9900.000000
1,Corporate,4,46800,11700.000000
2,Retail,6,29600,4933.333333
3,SME,4,27200,6800.000000


**Q14.** Corporate customers generated the most segment sales (KSh 46,800 from 4 orders) and had the highest average order value (KSh 11,700). Retail generated KSh 29,600 from 6 orders, the largest number of orders, but a lower average value of KSh 4,933.33. SME generated KSh 27,200 from 4 orders; the unmatched C999 order contributes KSh 9,900 to the NULL segment and should be investigated rather than attributed to a known segment.

## Part K — Merge the JSON region lookup

In [11]:
regions_df = pd.read_json("regions.json")
customers_regions = customers_clean.merge(regions_df, on="RegionCode", how="left", validate="many_to_one")
display(customers_regions)
assert customers_regions["RegionName"].notna().all()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru
5,C006,Faith,SME,NRB,Nairobi
6,C007,George,Retail,MSA,Mombasa
7,C008,Halima,Corporate,KSM,Kisumu
8,C009,Ian,Retail,NKR,Nakuru
9,C010,Jane,SME,NRB,Nairobi


**Q15.** Yes. The four customer RegionCodes map to readable region names (Nairobi, Mombasa, Kisumu, Nakuru).

**Q16.** A left merge retains the customer row and supplies a missing/NULL `RegionName` when its RegionCode has no match in the lookup.

## Final challenge — Regional sales analysis

In [12]:
# Use the validated JSON lookup in pandas, then aggregate at order grain.
order_regions = clean_joined.merge(regions_df, on="RegionCode", how="left", validate="many_to_one")
regional_kpis = (order_regions.assign(Region=order_regions["RegionName"].fillna("Unmatched / NULL"))
                 .groupby("Region", as_index=False)
                 .agg(NumberOfOrders=("OrderID", "count"), TotalSales=("OrderAmount", "sum"))
                 .sort_values("Region"))
display(regional_kpis)

,Region,NumberOfOrders,TotalSales
0,Kisumu,2,20100
1,Mombasa,3,23300
2,Nairobi,7,39800
3,Nakuru,2,20400
4,Unmatched / NULL,1,9900


**Q17.** Nairobi generated the highest identified-region sales: KSh 39,800 across 7 orders.

**Q18.** No. Revenue alone omits costs and profit/margins, so high sales may not mean high profitability. The dataset also lacks market-size or exposure context, customer acquisition/retention and service costs, and a time series; only 15 orders are represented, and one order (C999, KSh 9,900) cannot be assigned to a region.

## Short conclusion

The most important data-quality issue was a duplicate customer key, C004, which made a LEFT JOIN duplicate O005 and overstate sales by KSh 9,100. Removing the duplicate after documenting it restored the join to 15 rows and preserved the KSh 113,500 source total, while the unmatched C999 order remained visible. Totals should be reconciled after joins because row multiplication or record loss can silently distort business KPIs. Corporate generated the most sales at KSh 46,800. This small dataset contains no cost/profit or market-size data, and the unmatched order cannot be regionally attributed.

## Reproducibility and submission checklist

The notebook regenerates `customers.csv`, `orders.csv`, `regions.json`, and `retail_lab.db` on execution. It uses relative paths and can be restarted and run from top to bottom. Before GitHub submission, replace the README name/student ID placeholders, rename the repository to `DSA2050-Week2-<your-student-ID>`, and publish it under your GitHub account.